This notebook processes ridership data from multiple transit agencies to create representative stop-level datasets for weekdays, Saturdays, and Sundays. For each agency, the workflow generally includes:

- Adding day type based on the service date (Weekday, Saturday, Sunday), with holidays normalized or removed as needed.
- Cleaning and renaming columns to ensure consistency across datasets (stop IDs, stop names, route names, and ridership fields).
- Summing boardings and alightings across directions at each stop, and aggregating multiple time periods where applicable.
- Calculating weighted averages when ridership is reported over multiple periods, using the number of service days as weights.
- Setting a ridership basis flag to indicate that values represent reported daily boardings and/or alightings.
- Computing final average ridership per stop, route, and day type, producing clean, representative stop-level datasets ready for modeling, analysis, or comparison across agencies.

This approach standardizes ridership data across agencies with differing data formats, missing values, and service coverage, enabling consistent cross-agency analysis.

In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import numpy as np
import google.auth
import os
import gcsfs
from calitp_data_analysis.sql import get_engine
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
from pandas.tseries.holiday import USFederalHolidayCalendar
fs = gcsfs.GCSFileSystem()

pd.set_option('display.max_columns', None)

In [3]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships'

In [4]:
bart_ridership = pd.read_excel(f"{GCS_FILE_PATH}/bart_ridership.xlsx")
big_blue_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/big_blue_bus_ridership.xlsx")
caltrain_ridership = pd.read_excel(f"{GCS_FILE_PATH}/caltrain_ridership.xlsx")
culver_citybus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/culver_citybus_ridership.xlsx")
foothill_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/foothill_transit_ridership.xlsx")
fresno_area_express_ridership = pd.read_excel(f"{GCS_FILE_PATH}/fresno_area_express_ridership.xlsx")
# gold_coast_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/gold_coast_transit_ridership.xlsx")
golden_gate_park_shuttle_ridership = pd.read_excel(f"{GCS_FILE_PATH}/golden_gate_park_shuttle_ridership.xlsx")
golden_gate_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/golden_gate_transit_ridership.xlsx")
long_beach_transit_ridership=  pd.read_excel(f"{GCS_FILE_PATH}/long_beach_transit_ridership.xlsx")
# octa_ridership = pd.read_excel(f"{GCS_FILE_PATH}/octa_ridership.xlsx")
# omni_trans_ridership= pd.read_excel(f"{GCS_FILE_PATH}/omni_trans_ridership.xlsx")
riverside_transit_ridership= pd.read_excel(f"{GCS_FILE_PATH}/riverside_transit_ridership.xlsx")
# sacrt_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sacrt_bus_ridership.xlsx")
samtrans_ridership = pd.read_excel(f"{GCS_FILE_PATH}/samtrans_ridership.xlsx")
# santa_cruz_metro_ridership = pd.read_excel(f"{GCS_FILE_PATH}/santa_cruz_metro_ridership.xlsx")
# sbmtd_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sbmtd_ridership.xlsx")
sdmts_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sdmts_ridership.xlsx")
# sunline_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sunline_transit_ridership.xlsx")


In [5]:
def add_day_type(df, date_col, output_col='day_type'):
    """
    Classify each row as 'Weekday', 'Saturday', or 'Sunday' based on a specified date column.
    This is useful when daily datasets include labels such as 'Holiday' or when no day_type
    column exists. The function converts the date column, identifies the day of week, and
    standardizes the day-type classification accordingly. Either a start_date or end_date
    column can be used for datasets with daily reporting.
    """
    df = df.copy()

    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    dow = df[date_col].dt.dayofweek  

    df[output_col] = np.select(
        condlist=[dow.eq(5), dow.eq(6)],
        choicelist=['Saturday', 'Sunday'],
        default='Weekday'
    )

    return df

In [6]:
# Counting number of days without holidays
def count_service_days(row):
    dates = pd.date_range(row.start_date, row.end_date)

    if row.day_type == "Weekday":
        return sum(d.weekday() < 5 for d in dates)
    elif row.day_type == "Saturday":
        return sum(d.weekday() == 5 for d in dates)
    elif row.day_type == "Sunday":
        return sum(d.weekday() == 6 for d in dates)

In [7]:
# Counting number of days with holidays
def count_service_days_wo_holidays(row):
    dates = pd.date_range(row.start_date, row.end_date)

    if row.day_type == "Weekday":
        return sum((d.weekday() < 5) and (d not in holiday_set) for d in dates)

    elif row.day_type == "Saturday":
        return sum((d.weekday() == 5) and (d not in holiday_set) for d in dates)

    elif row.day_type == "Sunday":
        return sum((d.weekday() == 6) and (d not in holiday_set) for d in dates)

In [8]:
def compute_averages(df, ridership_cols, group_cols):
    """
    Computes average of one or more ridership columns per group in long format.

    Parameters:
    - df: pandas DataFrame
    - ridership_cols: list of columns to average (e.g., ['daily_boardings', 'daily_alightings'])
    - group_cols: list of columns to group by (e.g., ['stop_name', 'day_type'])

    Returns:
    - DataFrame with columns: group_cols + average of each ridership column
    """
    
    if isinstance(ridership_cols, str):
        ridership_cols = [ridership_cols]  # allow single string input
    
    averaged = df.groupby(group_cols)[ridership_cols].mean().reset_index()
    
    # rename each column to indicate it's averaged
    averaged = averaged.rename(columns={col: f"average_{col}" for col in ridership_cols})
    
    return averaged

In [9]:
bart_ridership = add_day_type(bart_ridership, date_col='start_date')
bart_ridership['daily_boardings'] = pd.to_numeric(bart_ridership['daily_boardings'], errors='coerce')
bart_ridership['daily_alightings'] = pd.to_numeric(bart_ridership['daily_alightings'], errors='coerce')
average_bart_ridership = compute_averages(
    df=bart_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["dataset_id", "organization_name", "service_name", 
                "stop_id", "stop_name", "stop_lat", "stop_lon", "day_type"]
    
bart_ridership_final = average_bart_ridership[['organization_name', 'service_name', 'stop_id', 
                                               'stop_name', 'day_type', 'average_daily_boardings',
                                              'average_daily_alightings']]

,dataset_id,organization_name,service_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
142,011CF30F49575609,San Francisco Bay Area Rapid Transit District,Bay Area Rapid Transit,909209,Warm Springs / South Fremont,37.502285,-121.939395,Sunday,425.538462,436.519231
149,011CF30F49575609,San Francisco Bay Area Rapid Transit District,Bay Area Rapid Transit,909509,Berryessa,37.368473,-121.874681,Weekday,1447.413793,1497.176245
70,011CF30F49575609,San Francisco Bay Area Rapid Transit District,Bay Area Rapid Transit,903209,Orinda,37.878481,-122.183667,Sunday,486.711538,494.865385
127,011CF30F49575609,San Francisco Bay Area Rapid Transit District,Bay Area Rapid Transit,906409,Millbrae,37.600237,-122.386757,Sunday,1065.346154,1282.115385
6,011CF30F49575609,San Francisco Bay Area Rapid Transit District,Bay Area Rapid Transit,900309,MacArthur,37.828803,-122.267105,Saturday,2427.903846,2340.846154


In [10]:
big_blue_bus_ridership.columns = big_blue_bus_ridership.columns.str.lower()
big_blue_bus_ridership['service_day'] = big_blue_bus_ridership['service_day'].str.strip().str.capitalize()


big_blue_bus_ridership = big_blue_bus_ridership.rename(columns={
    'route_number': 'route_short_name',
    'service_day': 'day_type',
    'average_daily_boardings':'average_daily_boardings_period',
    'average_daily_alightings':'average_daily_alightings_period'    
})


# Sum boardings/alightings across directions within same stop & period
total_big_blue_bus_ridership = (
    big_blue_bus_ridership
    .groupby(['day_type','route_short_name','route_name',
              'stop_id','stop_name','stop_lat','stop_lon',
              'start_date','end_date'])
    .agg({
        'average_daily_boardings_period':'sum',
        'average_daily_alightings_period':'sum'
    })
    .reset_index()
)


In [11]:
big_blue_bus_ridership['start_date'] = pd.to_datetime(big_blue_bus_ridership['start_date'])
big_blue_bus_ridership['end_date'] = pd.to_datetime(big_blue_bus_ridership['end_date'])

total_big_blue_bus_ridership["n_days"] = total_big_blue_bus_ridership.apply(count_service_days, axis=1)

# Taking weighted average of the ridership across 4 different time periods
averaged_total_big_blue_bus_ridership = (
    total_big_blue_bus_ridership
    .groupby(['day_type','route_short_name','route_name',
              'stop_id','stop_lat','stop_lon'])
    .apply(lambda x: pd.Series({
        'avg_daily_boardings': np.average(x['average_daily_boardings_period'], weights=x['n_days']),
        'avg_daily_alightings': np.average(x['average_daily_alightings_period'], weights=x['n_days'])
    }))
    .reset_index()
)


# Set the ridership basis flag
averaged_total_big_blue_bus_ridership["daily_ridership_basis"] = "reported_avg_daily"
averaged_total_big_blue_bus_ridership["organization_name"] = "Big Blue Bus"


averaged_total_big_blue_bus_ridership.sample(5)

/tmp/ipykernel_1451/2421701129.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,day_type,route_short_name,route_name,stop_id,stop_lat,stop_lon,avg_daily_boardings,avg_daily_alightings,daily_ridership_basis
394,Saturday,8,Ocean Park Blvd & Westwood Bl/UCLA,2138,34.022770,-118.438309,19.247857,7.636213,reported_avg_daily
2515,Weekday,15,Barrington Ave,2860,34.034672,-118.448394,1.096385,4.061001,reported_avg_daily
477,Saturday,9,Pacific Palisades,2737,34.049698,-118.529614,4.526718,0.024259,reported_avg_daily
2935,Weekday,43,San Vicente Blvd & 26th St,2898,34.026274,-118.473629,0.566197,1.177086,reported_avg_daily
1019,Sunday,3,Lincoln Blvd/LAX,2196,33.959978,-118.396710,114.893549,24.053482,reported_avg_daily


In [12]:
caltrain_ridership = caltrain_ridership.rename(columns={
    'Origin Station': 'stop_name',
    'Date Type': 'day_type',
    'Average Ridership':'average_daily_boardings_period',  
})

caltrain_ridership = caltrain_ridership[
    caltrain_ridership["day_type"] != "Holiday"
]

# Getting number of days and using US Federal Holidays
start_date_caltrain = '2023-11-01'
end_date_caltrain = '2025-07-31'

#Getting all the federal holidays between the start and end date
cal = USFederalHolidayCalendar()
holidays = cal.holidays(start=start_date_caltrain, end=end_date_caltrain)

# Setting holidays to calculate the number of weekdays, saturday and sunday without holidays 
holiday_set = set(holidays)

caltrain_ridership["start_date"] = pd.to_datetime(caltrain_ridership["start_date"])
caltrain_ridership["end_date"] = pd.to_datetime(caltrain_ridership["end_date"])

caltrain_ridership["n_days"] = caltrain_ridership.apply(count_service_days_wo_holidays, axis=1)

In [13]:
# Taking weighted average of the ridership across 20 months time periods
averaged_caltrain_ridership = (
    caltrain_ridership
    .groupby(['day_type','stop_name'])
    .apply(lambda x: pd.Series({
        'avg_daily_boardings': np.average(x['average_daily_boardings_period'], weights=x['n_days'])
    }))
    .reset_index()
)

# Set the ridership basis flag
averaged_caltrain_ridership["daily_ridership_basis"] = "reported_avg_daily"
averaged_caltrain_ridership["organization_name"] = "Caltrain"

averaged_caltrain_ridership.sample(5)

/tmp/ipykernel_1451/1226124597.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,day_type,stop_name,avg_daily_boardings,daily_ridership_basis
77,Weekday,Palo Alto,3139.693563,reported_avg_daily
29,Saturday,Tamien,58.297552,reported_avg_daily
30,Sunday,22nd Street,354.256197,reported_avg_daily
83,Weekday,San Jose Diridon,1789.561827,reported_avg_daily
88,Weekday,Sunnyvale,1408.195136,reported_avg_daily


In [14]:
culver_citybus_ridership["daily_ridership_basis"] = "reported_avg_daily"
culver_citybus_ridership["Route"] = culver_citybus_ridership["Route"].str.split("-", n=1).str[1]


culver_citybus_ridership = culver_citybus_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'route_id': 'route_short_name',
    'Route': 'route_name',
})

group_cols = [
    "route_name", "stop_id", "stop_name", "route_short_name", "day_type", "daily_ridership_basis"]

# Sum AVG On and AVG Off
culver_citybus_ridership = (culver_citybus_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("AVG On", "sum"),
        daily_alightings=("AVG Off", "sum")
    )
)

culver_citybus_ridership["organization_name"] = "Culver City Bus"

culver_citybus_ridership.sample(5)

,route_name,stop_id,stop_name,route_short_name,day_type,daily_ridership_basis,daily_boardings,daily_alightings
806,Washington Boulevard,106,Washington Blvd/Palawan Way,1,Saturday,reported_avg_daily,37.9,4.5
770,Sepulveda Boulevard,696,WESTCHESTER/AIRPORT,6,Saturday,reported_avg_daily,57.9,6.2
704,Sepulveda Boulevard,664,Sepulveda Blvd/Washington Pl,6,Weekday,reported_avg_daily,21.6,37.2
941,Washington Boulevard,151,WashingtonBlvd/ClaringtonAve,1,Saturday,reported_avg_daily,16.5,7.7
555,Sepulveda Boulevard,618,Center Dr/Park Terrace Dr,6,Sunday,reported_avg_daily,26.4,20.6


In [15]:
# Add day_type from the date column
foothill_transit_ridership = add_day_type(foothill_transit_ridership, date_col="date")

# Grouping columns
group_cols = [
    "date", "route_short_name", "stop_code", "stop_lat", "stop_lon",
    "start_date", "end_date", "day_type"
]

# Sum boardings and alightings, with explicit output names
total_ridership_foothill = (
    foothill_transit_ridership
    .groupby(group_cols, as_index=False)
    .agg(
        daily_boardings=("boardings", "sum"),
        daily_alightings=("alightings", "sum")
    )
)

# Set the ridership basis flag
total_ridership_foothill["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_ridership_foothill['daily_boardings'] = pd.to_numeric(total_ridership_foothill['daily_boardings'], errors='coerce')
total_ridership_foothill['daily_alightings'] = pd.to_numeric(total_ridership_foothill['daily_alightings'], errors='coerce')

average_foothill_ridership = compute_averages(
    df=total_ridership_foothill,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["route_short_name", "stop_code", 
                "stop_lat", "stop_lon", "day_type"]
)

average_foothill_ridership["organization_name"] = "Foothill Transit"
average_foothill_ridership.head(5)


,route_short_name,stop_code,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
0,178,23,34.034964,-117.919263,Saturday,1.689655,0.310345
1,178,23,34.034964,-117.919263,Sunday,1.750000,0.281250
2,178,23,34.034964,-117.919263,Weekday,2.144186,0.218605
3,178,24,34.035320,-117.919706,Saturday,2.341463,0.097561
4,178,24,34.035320,-117.919706,Sunday,2.375000,0.156250


In [16]:
# Add day_type from the date column
fresno_area_express_ridership = add_day_type(fresno_area_express_ridership, date_col="Date")

fresno_area_express_ridership["daily_ridership_basis"] = "reported_daily"


fresno_area_express_ridership = fresno_area_express_ridership.rename(columns={
    'StopID': 'stop_id',
    'StopLabel': 'stop_name',
    'ProjectedBoarding': 'daily_boardings',
    'ProjectedAlighting': 'daily_alightings'
})

# Calculate average ridership per day type and stop
fresno_area_express_ridership['daily_boardings'] = pd.to_numeric(fresno_area_express_ridership['daily_boardings'], errors='coerce')
fresno_area_express_ridership['daily_alightings'] = pd.to_numeric(fresno_area_express_ridership['daily_alightings'], errors='coerce')

average_fresno_area_express_ridership = compute_averages(
    df=fresno_area_express_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["stop_id",  "stop_name", "day_type"]
)

average_fresno_area_express_ridership["organization_name"] = "Fresno County"

average_fresno_area_express_ridership.head(5)

,stop_id,stop_name,day_type,average_daily_boardings,average_daily_alightings
0,5,NE BRAWLEY - SHIELDS,Saturday,35.329309,25.519283
1,5,NE BRAWLEY - SHIELDS,Sunday,33.548114,25.662840
2,5,NE BRAWLEY - SHIELDS,Weekday,51.142125,29.915370
3,6,SE SHAW - BRAWLEY,Saturday,7.004770,4.812710
4,6,SE SHAW - BRAWLEY,Sunday,5.616939,3.298467


In [17]:
# Add day_type from the date column
golden_gate_park_shuttle_ridership = add_day_type(golden_gate_park_shuttle_ridership, date_col="Date")

group_cols = [
    "Date", "day_type", "Stop", "start_date", "end_date"]

# Sum AVG On 
total_golden_gate_park_shuttle_ridership = (golden_gate_park_shuttle_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("Ridership", "sum"),
    )
)

total_golden_gate_park_shuttle_ridership["daily_ridership_basis"] = "reported_daily"

total_golden_gate_park_shuttle_ridership = total_golden_gate_park_shuttle_ridership.rename(columns={
    'Stop': 'stop_name',
    'Date': 'date'
})

# Calculate average ridership per day type and stop
total_golden_gate_park_shuttle_ridership['daily_boardings'] = pd.to_numeric(total_golden_gate_park_shuttle_ridership['daily_boardings'], errors='coerce')
average_golden_gate_park_shuttle_ridership = compute_averages(
    df=total_golden_gate_park_shuttle_ridership,
    ridership_cols=["daily_boardings"],
    group_cols=["stop_name", "day_type"]
)


average_golden_gate_park_shuttle_ridership["organization_name"] = "Golden Gate Park Shuttle"
average_golden_gate_park_shuttle_ridership.head(5)

,stop_name,day_type,average_daily_boardings
0,10th Ave/ De Young EB,Saturday,9.211538
1,10th Ave/ De Young EB,Sunday,7.076923
2,10th Ave/ De Young EB,Weekday,5.862069
3,10th Ave/ De Young WB,Saturday,9.519231
4,10th Ave/ De Young WB,Sunday,8.442308


In [18]:
# Add day_type from the date column
golden_gate_transit_ridership = add_day_type(golden_gate_transit_ridership, date_col="date")

golden_gate_transit_ridership = golden_gate_transit_ridership.rename(columns={
    'STOP_NUMBER': 'stop_id',
    'STOP_NAME': 'stop_name',
    'ROUTE': 'route_short_name',
})

# Remove everything in parentheses (including the parentheses) at the end of the string
golden_gate_transit_ridership['stop_name'] = golden_gate_transit_ridership['stop_name'].str.replace(r'\s*\(.*\)$', '', regex=True)

group_cols = [
    "day_type", "route_short_name", "stop_id", "stop_name", "end_date", "start_date"]

# Sum AVG On and AVG Off
total_golden_gate_transit_ridership = (golden_gate_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("BOARDINGS", "sum"),
        daily_alightings=("ALIGHTINGS", "sum")
    )
)

total_golden_gate_transit_ridership["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_golden_gate_transit_ridership['daily_boardings'] = pd.to_numeric(total_golden_gate_transit_ridership['daily_boardings'], errors='coerce')
total_golden_gate_transit_ridership['daily_alightings'] = pd.to_numeric(total_golden_gate_transit_ridership['daily_alightings'], errors='coerce')

average_golden_gate_transit_ridership = compute_averages(
    df=total_golden_gate_transit_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["route_short_name", "stop_id", "stop_name", "day_type"]
)

average_golden_gate_transit_ridership["organization_name"] = "Golden Gate Transit"



average_golden_gate_transit_ridership.head(5)

,route_short_name,stop_id,stop_name,day_type,average_daily_boardings,average_daily_alightings
0,101,40003,Salesforce Transit Center-Bus Plaza Bay A,Saturday,47.000000,0.00
1,101,40003,Salesforce Transit Center-Bus Plaza Bay A,Sunday,35.250000,0.00
2,101,40003,Salesforce Transit Center-Bus Plaza Bay A,Weekday,49.363636,0.00
3,101,40023,Golden Gate Ave & Polk St,Saturday,0.250000,37.25
4,101,40023,Golden Gate Ave & Polk St,Sunday,0.500000,35.00


In [19]:
long_beach_transit_ridership["daily_ridership_basis"] = "reported_avg_daily"

group_cols = [
    "DayType", "Route", "StopID", "StopName", "end_date", "start_date", "daily_ridership_basis"]

# Sum AVG On and AVG Off
total_long_beach_transit_ridership = (long_beach_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("Boardings", "sum"),
        daily_alightings=("Alightings", "sum")
    )
)

total_long_beach_transit_ridership = total_long_beach_transit_ridership.rename(columns={
    'StopID': 'stop_id',
    'StopName': 'stop_name',
    'DayType': 'day_type',
    'Route': 'route_short_name',
})

total_long_beach_transit_ridership["organization_name"] = "Long Beach Transit"


total_long_beach_transit_ridership.head(5)

,day_type,route_short_name,stop_id,stop_name,end_date,start_date,daily_ridership_basis,daily_boardings,daily_alightings
0,Saturday,1,2001,2727 Del Amo Blvd N,2025-06-30,2024-07-01,reported_avg_daily,2.582828,0.000000
1,Saturday,1,2002,2660 Del Amo Blvd S,2025-06-30,2024-07-01,reported_avg_daily,0.000000,0.000000
2,Saturday,1,2003,Del Amo & Rancho Way NW,2025-06-30,2024-07-01,reported_avg_daily,2.282203,4.831939
3,Saturday,1,2004,Del Amo & Fordyce SW,2025-06-30,2024-07-01,reported_avg_daily,5.977199,2.219394
4,Saturday,1,2005,Del Amo & Wilmington NW,2025-06-30,2024-07-01,reported_avg_daily,3.527042,5.859201


In [20]:
riverside_transit_ridership = add_day_type(riverside_transit_ridership, date_col='date')

group_cols = [
    "date", "Stop ID", "Route", "Longitude", "Latitude", "start_date", 
    "end_date", "day_type"]

# Sum AVG On and AVG Off
total_riverside_transit_ridership = (riverside_transit_ridership.groupby(
    group_cols, as_index=False).agg(
    daily_boardings = ('ridership', 'sum'))
)

total_riverside_transit_ridership = total_riverside_transit_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Route': 'route_short_name',
    'Latitude': 'stop_lat',
    'Longitude': 'stop_lon'

})

total_riverside_transit_ridership["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_riverside_transit_ridership['daily_boardings'] = pd.to_numeric(total_riverside_transit_ridership['daily_boardings'], errors='coerce')
average_riverside_transit_ridership = compute_averages(
    df=total_riverside_transit_ridership,
    ridership_cols=["daily_boardings"],
    group_cols=["route_short_name", "stop_id", "stop_lat", "stop_lon", "day_type"]
)

total_riverside_transit_ridership["organization_name"] = "Riverside Transit"

average_riverside_transit_ridership.sample(5)

,route_short_name,stop_id,stop_lat,stop_lon,day_type,average_daily_boardings
84386,12,1148,33.979772,-117.375624,Weekday,3.0
290087,29,2221,33.977340,-117.462672,Weekday,1.0
200115,19,3486,33.916352,-117.226496,Weekday,7.0
96412,12,1573,34.001024,-117.364288,Weekday,2.0
268992,27,2749,33.882300,-117.369824,Weekday,5.0


In [21]:
samtrans_ridership = add_day_type(samtrans_ridership, date_col='APC Date')

samtrans_ridership = samtrans_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Route': 'route_short_name',
    'Lat': 'stop_lat',
    'Lon': 'stop_lon',
    'APC Date': 'date',
    'Ons': 'daily_boardings',
    'Offs': 'daily_alightings',
})

samtrans_ridership["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
samtrans_ridership['daily_boardings'] = pd.to_numeric(samtrans_ridership['daily_boardings'], errors='coerce')
samtrans_ridership['daily_alightings'] = pd.to_numeric(samtrans_ridership['daily_alightings'], errors='coerce')

average_samtrans_ridership = compute_averages(
    df=samtrans_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["route_short_name", "stop_id", "stop_lat", "stop_lon", "day_type"]
)

average_samtrans_ridership["organization_name"] = "Samtrans"

average_samtrans_ridership.sample(5)

,route_short_name,stop_id,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
3579,294,341042,37.541328,-122.314926,Saturday,2.0,0.0
3127,281,363065,37.468845,-122.141273,Sunday,8.0,0.4
4682,397,343059,37.494075,-122.243743,Saturday,1.0,0.5
3275,292,334008,37.665744,-122.391235,Saturday,32.4,42.6
1448,122,334238,37.637172,-122.450279,Saturday,2.2,5.2


In [22]:
group_cols = [
    "Route", "Route Name", "Stop ID",  "Stop Name", "day_type", "start_date", 
    "end_date"]

# Sum AVG On and AVG Off
total_sdmts_ridership = (sdmts_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("Average On", "sum"),
        daily_alightings=("Average Off", "sum")
    )
)

total_sdmts_ridership["Route Name"] = total_sdmts_ridership["Route Name"].str.split(":", n=1).str[1]

total_sdmts_ridership = total_sdmts_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Route': 'route_short_name',
    'Route Name': 'route_name',
    'Stop Name': 'stop_name',
})

total_sdmts_ridership["daily_ridership_basis"] = "reported_avg_daily"

total_sdmts_ridership["organization_name"] = "SDMTS"

total_sdmts_ridership.head(5)

,route_short_name,route_name,stop_id,stop_name,day_type,start_date,end_date,daily_boardings,daily_alightings,daily_ridership_basis
0,1,Fashion Valley-La Mesa,10106,University Av & 10th Av,Saturday,2024-09-01,2025-01-25,16.513085,8.709722,reported_avg_daily
1,1,Fashion Valley-La Mesa,10106,University Av & 10th Av,Sunday,2024-09-01,2025-01-25,14.264056,8.324885,reported_avg_daily
2,1,Fashion Valley-La Mesa,10106,University Av & 10th Av,Weekday,2024-09-01,2025-01-25,26.228026,15.629832,reported_avg_daily
3,1,Fashion Valley-La Mesa,10111,University Av & Vermont St,Saturday,2024-09-01,2025-01-25,42.220322,14.795541,reported_avg_daily
4,1,Fashion Valley-La Mesa,10111,University Av & Vermont St,Sunday,2024-09-01,2025-01-25,29.279929,10.192920,reported_avg_daily
